# Phase 1 — Step 2: Curation Pipeline
Input: `thesis-raw-images` dataset (zip)

Output: `/kaggle/working/curated/` — ảnh đã chuẩn hóa 224x224

**Pipeline:**
1. Giải nén zip
2. pHash deduplication
3. Resize → center crop 224x224
4. Integrity check
5. Lưu log + zip output

## ⚙️ Cell 1 — Config

In [9]:
from pathlib import Path

# ── Input — trỏ thẳng vào folder, không cần giải nén ─────────────────────────
EXTRACT_DIR = Path('/kaggle/input/datasets/chngnguynvhong/thesis-raw-images/thesis_raw_images/raw')

# ── Output dirs ───────────────────────────────────────────────────────────────
CURATED_DIR = Path('/kaggle/working/curated')
LOG_DIR     = Path('/kaggle/working/logs')

for d in [CURATED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Curation config ───────────────────────────────────────────────────────────
TARGET_SIZE     = 224
PHASH_THRESHOLD = 3
MIN_SIZE_BEFORE = 100

TARGET_COUNTS = {
    'real':     8000,
    'fake_sd':  3000,
    'fake_gan': 2000,
    'fake_mj':  2000,
}

print('✓ Config OK')
print(f'  EXTRACT_DIR: {EXTRACT_DIR}')
print(f'  TARGET_SIZE: {TARGET_SIZE}x{TARGET_SIZE}')

✓ Config OK
  EXTRACT_DIR: /kaggle/input/datasets/chngnguynvhong/thesis-raw-images/thesis_raw_images/raw
  TARGET_SIZE: 224x224


## 📦 Cell 2 — Giải nén zip

In [10]:
# Kiểm tra cấu trúc — không cần giải nén
print('Cấu trúc raw:')
if not EXTRACT_DIR.exists():
    print(f'[ERROR] {EXTRACT_DIR} không tồn tại!')
    print('        Kiểm tra lại tên dataset đã add')
else:
    for folder in sorted(EXTRACT_DIR.iterdir()):
        if not folder.is_dir(): continue
        imgs = list(folder.glob('*.jpg')) + list(folder.glob('*.png'))
        print(f'  {folder.name:15s} {len(imgs):,} ảnh')

Cấu trúc raw:
  fake_gan        2,000 ảnh
  fake_mj         2,000 ảnh
  fake_sd         3,000 ảnh
  real            8,000 ảnh


## 🔍 Cell 3 — Helper functions
pHash dedup + center crop 224x224

In [11]:
import numpy as np
import json, random, shutil
from PIL import Image

random.seed(42)
curation_log = {}

# ── pHash ─────────────────────────────────────────────────────────────────────
def phash(img: Image.Image, hash_size: int = 8) -> str:
    """Perceptual hash dùng average hash — đơn giản, ít false positive hơn."""
    img  = img.convert('L').resize((hash_size, hash_size), Image.LANCZOS)
    arr  = np.array(img, dtype=float)
    mean = arr.mean()
    bits = (arr > mean).flatten()
    return ''.join('1' if b else '0' for b in bits)

def hamming(h1: str, h2: str) -> int:
    return sum(c1 != c2 for c1, c2 in zip(h1, h2))

# ── Center crop 224x224 ───────────────────────────────────────────────────────
def center_crop(img: Image.Image, size: int = TARGET_SIZE) -> Image.Image:
    """
    Resize cạnh ngắn về size, rồi center crop size x size.
    Không distort aspect ratio — giữ nguyên frequency pattern.
    """
    img = img.convert('RGB')
    w, h = img.size

    # Resize cạnh ngắn về target_size
    if w < h:
        new_w, new_h = size, int(h * size / w)
    else:
        new_w, new_h = int(w * size / h), size

    img  = img.resize((new_w, new_h), Image.LANCZOS)
    left = (new_w - size) // 2
    top  = (new_h - size) // 2
    return img.crop((left, top, left + size, top + size))

print('✓ Helper functions loaded')

✓ Helper functions loaded


## 🧹 Cell 4 — Dedup + Normalize: Real images

In [12]:
def curate_class(label: str, target: int):
    src_dir = EXTRACT_DIR / label
    out_dir = CURATED_DIR / label
    out_dir.mkdir(parents=True, exist_ok=True)

    # Skip nếu đã curate đủ
    existing = list(out_dir.glob('*.png'))
    if len(existing) >= target:
        print(f'  [{label}] Đã có {len(existing)} ảnh — skip')
        return len(existing)

    all_imgs = list(src_dir.glob('*.jpg')) + list(src_dir.glob('*.png'))
    random.shuffle(all_imgs)
    print(f'  [{label}] {len(all_imgs):,} ảnh nguồn → target {target}')

    seen_hashes = {}
    kept = 0
    dedup_removed = 0
    size_removed  = 0

    for img_path in tqdm(all_imgs, desc=f'  {label}', leave=False):
        if kept >= target:
            break
        try:
            img = Image.open(img_path).convert('RGB')
            w, h = img.size

            # Lọc ảnh quá nhỏ
            if w < MIN_SIZE_BEFORE or h < MIN_SIZE_BEFORE:
                size_removed += 1
                continue

            # pHash dedup
            h_str = phash(img)
            is_dup = any(
                hamming(h_str, existing_h) <= PHASH_THRESHOLD
                for existing_h in seen_hashes.values()
            )
            if is_dup:
                dedup_removed += 1
                continue

            # Center crop 224x224
            img_cropped = center_crop(img, TARGET_SIZE)

            # Lưu PNG
            out_path = out_dir / f'{label}_{kept:05d}.png'
            img_cropped.save(out_path, format='PNG', compress_level=1)

            seen_hashes[str(img_path)] = h_str
            kept += 1

        except Exception:
            continue

    print(f'  [{label}] ✓ kept={kept}, dedup={dedup_removed}, size_filter={size_removed}')
    return kept

# Chạy real
n = curate_class('real', TARGET_COUNTS['real'])
curation_log['real'] = n

  [real] 8,000 ảnh nguồn → target 8000


  real:   0%|          | 0/8000 [00:00<?, ?it/s]

  [real] ✓ kept=7557, dedup=443, size_filter=0


## 🎨 Cell 5 — Dedup + Normalize: Fake SD

In [13]:
n = curate_class('fake_sd', TARGET_COUNTS['fake_sd'])
curation_log['fake_sd'] = n

  [fake_sd] 3,000 ảnh nguồn → target 3000


  fake_sd:   0%|          | 0/3000 [00:00<?, ?it/s]

  [fake_sd] ✓ kept=2967, dedup=33, size_filter=0


## 🤖 Cell 6 — Dedup + Normalize: Fake GAN

In [14]:
n = curate_class('fake_gan', TARGET_COUNTS['fake_gan'])
curation_log['fake_gan'] = n

  [fake_gan] 2,000 ảnh nguồn → target 2000


  fake_gan:   0%|          | 0/2000 [00:00<?, ?it/s]

  [fake_gan] ✓ kept=1894, dedup=106, size_filter=0


## 🖌️ Cell 7 — Dedup + Normalize: Fake MJ

In [15]:
n = curate_class('fake_mj', TARGET_COUNTS['fake_mj'])
curation_log['fake_mj'] = n

  [fake_mj] 2,000 ảnh nguồn → target 2000


  fake_mj:   0%|          | 0/2000 [00:00<?, ?it/s]

  [fake_mj] ✓ kept=1955, dedup=45, size_filter=0


## ✅ Cell 8 — Integrity check

In [16]:
print('Integrity check...')
all_ok = True

for folder in sorted(CURATED_DIR.iterdir()):
    if not folder.is_dir(): continue
    imgs   = list(folder.glob('*.png'))
    errors = []

    for p in tqdm(imgs, desc=f'Check {folder.name}', leave=False):
        try:
            img = Image.open(p)
            assert img.size == (TARGET_SIZE, TARGET_SIZE), f'Size sai: {img.size}'
            assert img.mode == 'RGB', f'Mode sai: {img.mode}'
        except Exception as e:
            errors.append(str(e))
            p.unlink()  # xóa ảnh lỗi

    status = '✓' if not errors else '✗'
    if errors: all_ok = False
    print(f'  {status} {folder.name:15s} {len(imgs):,} ảnh  errors={len(errors)}')

print()
if all_ok:
    print('✅ Integrity check passed')
else:
    print('⚠  Có lỗi — ảnh lỗi đã bị xóa tự động')

Integrity check...


Check fake_gan:   0%|          | 0/1894 [00:00<?, ?it/s]

  ✓ fake_gan        1,894 ảnh  errors=0


Check fake_mj:   0%|          | 0/1955 [00:00<?, ?it/s]

  ✓ fake_mj         1,955 ảnh  errors=0


Check fake_sd:   0%|          | 0/2967 [00:00<?, ?it/s]

  ✓ fake_sd         2,967 ảnh  errors=0


Check real:   0%|          | 0/7557 [00:00<?, ?it/s]

  ✓ real            7,557 ảnh  errors=0

✅ Integrity check passed


## 📊 Cell 9 — Tổng kết + lưu log

In [17]:
print('=' * 50)
print('KẾT QUẢ STEP 2')
print('=' * 50)

total = 0
for label, target in TARGET_COUNTS.items():
    out_dir = CURATED_DIR / label
    actual  = len(list(out_dir.glob('*.png')))
    pct     = actual / target * 100
    status  = '✓' if actual >= target * 0.8 else '⚠'
    print(f'  {status} {label:15s} {actual:,}/{target:,} ({pct:.0f}%)')
    total  += actual
    curation_log[label] = actual

print(f'\n  TOTAL: {total:,} ảnh @ {TARGET_SIZE}x{TARGET_SIZE}')

(LOG_DIR / 'step2_curation_log.json').write_text(
    json.dumps(curation_log, indent=2)
)
print('\n✅ Step 2 hoàn thành — chạy Step 3 (split)')

KẾT QUẢ STEP 2
  ✓ real            7,557/8,000 (94%)
  ✓ fake_sd         2,967/3,000 (99%)
  ✓ fake_gan        1,894/2,000 (95%)
  ✓ fake_mj         1,955/2,000 (98%)

  TOTAL: 14,373 ảnh @ 224x224

✅ Step 2 hoàn thành — chạy Step 3 (split)


## 💾 Cell 10 — Zip + Save output

In [18]:
import zipfile

OUT_ZIP = Path('/kaggle/working/thesis_curated.zip')

all_files = [f for f in CURATED_DIR.rglob('*') if f.is_file()] + \
            [f for f in LOG_DIR.rglob('*') if f.is_file()]

print(f'Đang nén {len(all_files):,} files...')
with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in tqdm(all_files, desc='Nén'):
        zf.write(f, f.relative_to('/kaggle/working'))

print(f'✓ Xong: {OUT_ZIP.stat().st_size/1024**2:.0f} MB')
print('  → Save Version → New Dataset → thesis-curated')

Đang nén 14,374 files...


Nén:   0%|          | 0/14374 [00:00<?, ?it/s]

✓ Xong: 1205 MB
  → Save Version → New Dataset → thesis-curated
